# Capstone 2.C: Retrieval SLO

The SLO, the constraints and the levers are in the assignment. This notebook gives you the plumbing: a client, the library, and helpers that build an index, move the alias, save Tina's retriever and show what she receives.

Every decision is yours: which levers, which chunking, which retriever, which settings. Nothing below makes one for you.

In [ ]:
import json, os, pathlib, sys
sys.path.insert(0, '/opt/ara/lib')
for line in pathlib.Path('/home/elastic/env').read_text().splitlines():
    if '=' in line and not line.startswith('#'):
        k, v = line.split('=', 1)
        os.environ.setdefault(k.strip(), v.strip())

from elasticsearch import Elasticsearch
from ara_index import (Chunk, load_corpus, chunk_fixed, chunk_heading_aware, chunk_unit_preserving,
                       chunk_report, standard_mapping, index_chunks, _make_chunk)

es = Elasticsearch(os.environ['ES_URL'], api_key=os.environ['ES_API_KEY'], request_timeout=300)
docs = load_corpus('/home/elastic/corpus')
dev = json.loads(pathlib.Path('/home/elastic/dev-sets/dev-questions.json').read_text())
print('ES', es.info()['version']['number'], '|', len(docs), 'documents |', len(dev), 'dev questions')
print('cortex-corpus-live ->', list(es.indices.get_alias(name='cortex-corpus-live')))
print('ARA_EMBED_ID  =', os.environ['ARA_EMBED_ID'])
print('ARA_RERANK_ID =', os.environ['ARA_RERANK_ID'])
print('retriever.json:', pathlib.Path('/home/elastic/retriever.json').read_text())

## Your decisions

Set what you choose and leave the rest as it is. The helpers refuse to run on an unset value, so nothing happens by default.

In [ ]:
# Name for a new index, if you re-chunk or tune. The starting index stays as it is.
NEW_INDEX = None

# Your chunking for one document. Return a list of Chunk: from chunk_fixed, chunk_heading_aware
# or chunk_unit_preserving, or from your own split with make_chunk(doc, text, i).
def chunk(doc):
    raise NotImplementedError('Write chunk(doc) if you re-chunk.')

# index_options for the body field of a new index (the Tune lever), or None for the standard mapping.
INDEX_OPTIONS = None

# Tina's search template: a dict with a top-level 'retriever' or 'query' key and '{query_text}'
# where the question goes. Leave it None to keep the current retriever.json.
RETRIEVER = None

## Helpers

Run this cell once. Each helper does one mechanical step and prints what it did.

- `make_chunk(doc, text, i)`: one `Chunk` from your own split.
- `chunk_all(chunker)`: runs your chunker on every document and prints the chunk report and the count against the starting index.
- `build_index(name, chunks, index_options=None)`: creates the index with the standard mapping on `ARA_EMBED_ID` (plus `index_options` on `body`, if given) and indexes the chunks.
- `point_alias(index)`: moves `cortex-corpus-live` to `index` in one request.
- `save_retriever(template)`: checks the template runs against `cortex-corpus-live`, then writes `retriever.json`.
- `show_top5(question)`: prints the chunks Tina receives for one question through the saved `retriever.json`.

In [ ]:
BASE_COUNT = json.loads(pathlib.Path('/home/elastic/baseline.json').read_text())['chunk_count']
EMBED_ID = os.environ['ARA_EMBED_ID']

def _need(value, name):
    if value is None:
        raise ValueError(f'Set {name} first, under Your decisions.')
    return value

def _fill(template, text):
    return json.loads(json.dumps(template).replace('{query_text}', json.dumps(text)[1:-1]))

def make_chunk(doc, text, i):
    """One Chunk from your own split: the document's metadata, your text, and its position i."""
    return _make_chunk(doc, text, i)

def chunk_all(chunker):
    """Run chunker on every document; print the chunk report and the count against the starting index."""
    chunks = [c for d in docs for c in chunker(d)]
    print(chunk_report(chunks), f'| {len(chunks) / BASE_COUNT:.2f}x the starting index\'s {BASE_COUNT} chunks')
    return chunks

def build_index(name, chunks, index_options=None):
    """Create index `name` with the standard mapping on ARA_EMBED_ID, then index the chunks."""
    _need(name, 'NEW_INDEX')
    if es.indices.exists(index=name):
        raise ValueError(f'{name} already exists. Pick a new name, or delete it with es.indices.delete(index={name!r}).')
    body = standard_mapping(EMBED_ID)
    if index_options is not None:
        body['mappings']['properties']['body']['index_options'] = index_options
    es.indices.create(index=name, body=body)
    result = index_chunks(es, name, chunks, require_existing=True)
    es.indices.refresh(index=name)
    print(name, result, '| chunks in the index:', es.count(index=name)['count'])
    return result

def point_alias(index):
    """Move cortex-corpus-live to `index` in one request, so Tina never sees an empty alias."""
    _need(index, 'NEW_INDEX')
    current = list(es.indices.get_alias(name='cortex-corpus-live'))
    actions = [{'remove': {'index': i, 'alias': 'cortex-corpus-live'}} for i in current if i != index]
    actions.append({'add': {'index': index, 'alias': 'cortex-corpus-live'}})
    es.indices.update_aliases(actions=actions)
    print('cortex-corpus-live ->', list(es.indices.get_alias(name='cortex-corpus-live')))

def save_retriever(template):
    """Check that template runs against cortex-corpus-live, then write it to retriever.json."""
    _need(template, 'RETRIEVER')
    if not isinstance(template, dict) or not ('retriever' in template or 'query' in template):
        raise ValueError("RETRIEVER must be a dict with a top-level 'retriever' or 'query' key.")
    if '{query_text}' not in json.dumps(template):
        raise ValueError("RETRIEVER needs '{query_text}' where the question goes.")
    es.search(index='cortex-corpus-live', body={**_fill(template, 'test'), 'size': 1})
    pathlib.Path('/home/elastic/retriever.json').write_text(json.dumps(template, indent=2))
    print('retriever.json saved.')

def show_top5(question, k=5):
    """The top k chunks Tina receives for one question, through the saved retriever.json."""
    template = json.loads(pathlib.Path('/home/elastic/retriever.json').read_text())
    body = {**_fill(template, question), 'size': k, '_source': ['doc_id', 'body_text']}
    for n, h in enumerate(es.search(index='cortex-corpus-live', body=body)['hits']['hits'], 1):
        text = ' '.join(h['_source'].get('body_text', '').split())
        print(f"{n}. {h['_source'].get('doc_id')}  about {int(len(text.split()) * 1.3)} tokens  {text[:160]}")

## Apply your changes

Uncomment the lines for the levers you chose, then run the cell.

In [ ]:
# chunks = chunk_all(chunk)
# build_index(NEW_INDEX, chunks, INDEX_OPTIONS)
# point_alias(NEW_INDEX)
# save_retriever(RETRIEVER)

## Scratch

## Submit

Scores the dev set the way the check scores the held-out set. Run it after every change.

In [ ]:
!/home/elastic/.venv/bin/python /home/elastic/submit.py